In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from dvpio.read.image import read_czi
import spatialdata as sd
import lazyslide as zs
from wsidata import open_wsi
from abmil_NEW import load_checkpoint 

TISSUE_KEY = "tissue"
TILE_KEY = "tiles_224"
FEATURE_KEY = "features_h-optimus-0"
TILE_PX = 224
MODEL_NAME = "h-optimus-0"
num_workers = os.cpu_count() // 2

# LOADING CZI
def load_czi_as_wsi(czi_path: str):
    img = read_czi(czi_path, channels=0)

    img = img.astype("uint8")

    sdata = sd.SpatialData()
    sdata.images["slide"] = img

    return open_wsi(sdata, image_key="slide")


# PREPROCESSING
def preprocess_wsi(wsi):
    print("Tissue segmentation..."
    if TISSUE_KEY not in wsi.shapes:
        zs.pp.find_tissues(wsi, key_added=TISSUE_KEY)

    print("Tiling...")
    if TILE_KEY not in wsi.shapes:
        zs.pp.tile_tissues(
            wsi,
            tile_px=TILE_PX,
            key_added=TILE_KEY,
            tissue_key=TISSUE_KEY,
        )

    print("Feature extraction...")
    if FEATURE_KEY not in wsi.tables:
        zs.tl.feature_extraction(
            wsi,
            model=MODEL_NAME,
            tile_key=TILE_KEY,
            key_added=FEATURE_KEY,
            num_workers = num_workers
            amp=True, batch_size=512, device = "cuda"
        )

    return wsi


# FEATURES
def load_features(wsi):
    adata = wsi.tables[FEATURE_KEY]

    feats = torch.from_numpy(adata.X).float()
    tile_ids = np.array(adata.obs["tile_id"])

    return feats, tile_ids


# MODEL INFERENCE
def abmil_inference(model, feats, device):
    feats = feats.to(device)

    with torch.no_grad():
        logits, attention = model(feats)

    probs = torch.softmax(logits.squeeze().float(), dim=0).cpu().numpy()
    pred = int(np.argmax(probs))

    return logits, probs, attention.squeeze().cpu().numpy(), pred

# POSTPROCESSING
def get_top_k_tiles(wsi, tile_ids, attention, k=10):
    tile_table = wsi.tables[FEATURE_KEY].obs.copy()

    df = pd.DataFrame({
        "tile_id": tile_ids,
        "attention": attention
    })

    merged = df.merge(
        tile_table[["tile_id", "geometry"]],
        on="tile_id",
        how="inner"
    )

    if merged.empty:
        return merged

    return merged.sort_values("attention", ascending=False).head(k)


# PIPELINE ENTRYPOINT
def run_inference(wsi, checkpoint_path, slide_path, top_k=10):
    model, config, label_mapping, device = load_checkpoint(checkpoint_path)

    feats, tile_ids = load_features(wsi)

    if feats.shape[0] == 0:
        raise ValueError(f"No tiles found for slide: {slide_path}")

    logits, probs, attention, pred = abmil_inference(model, feats, device)

    top_tiles = get_top_k_tiles(
        wsi=wsi,
        tile_ids=tile_ids,
        attention=attention,
        k=top_k
    )

    return {
        "slide": slide_path,
        "prediction": pred,
        "probabilities": probs,
        "attention": attention,
        "tile_ids": tile_ids,
        "top_k_tiles": top_tiles,
    }

/opt/anaconda3/envs/MaxPlanck/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'lazyslide'

In [ ]:
import os
import json
import pandas as pd

def export_tiles_to_dvpio_geojson(
    top_tiles_df,
    output_path,
    slide_id="slide",
    score_key="attention"
):
    features = []

    for _, row in top_tiles_df.iterrows():
        geom = row["geometry"]

        if geom is None:
            continue

        try:
            geom_json = geom.__geo_interface__
        except Exception:
            continue

        features.append({
            "type": "Feature",
            "geometry": geom_json,
            "properties": {
                "tile_id": str(row.get("tile_id")),
                "score": float(row.get(score_key, 0.0)),
                "slide": slide_id
            }
        })

    geojson = {
        "type": "FeatureCollection",
        "features": features
    }

    with open(output_path, "w") as f:
        json.dump(geojson, f)

    return geojson

def run_batch_inference(
    czi_paths,
    checkpoint_path,
    top_k=10,
    preprocess=True,
    export_geojson=True,
    geojson_dir="dvpio_geojson"
):
    results = []

    # create export folder once
    if export_geojson:
        os.makedirs(geojson_dir, exist_ok=True)

    for slide_path in czi_paths:
        print(f"\nProcessing: {slide_path}")

        try:
            print("Loading slide")
            wsi = load_czi_as_wsi(slide_path)

            print("Preprocessing slide")
            if preprocess:
                wsi = preprocess_wsi(wsi)

            print("Running inference")
            result = run_inference(
                wsi=wsi,
                checkpoint_path=checkpoint_path,
                slide_path=slide_path,
                top_k=top_k
            )

            if export_geojson and result["top_k_tiles"] is not None:
                slide_name = os.path.splitext(
                    os.path.basename(slide_path)
                )[0]

                geojson_path = os.path.join(
                    geojson_dir,
                    f"{slide_name}.geojson"
                )

                export_tiles_to_dvpio_geojson(
                    top_tiles_df=result["top_k_tiles"],
                    output_path=geojson_path,
                    slide_id=slide_name
                )

                result["geojson_path"] = geojson_path

            results.append(result)

        except Exception as e:
            print(f"Failed on {slide_path}: {e}")

            results.append({
                "slide": slide_path,
                "error": str(e),
                "prediction": None,
                "probabilities": None,
                "top_k_tiles": None,
                "geojson_path": None,
            })

    summary_df = pd.DataFrame([
        {
            "slide": r["slide"],
            "prediction": r.get("prediction"),
            "geojson_path": r.get("geojson_path"),
            "error": r.get("error", None),
        }
        for r in results
    ])

    return results, summary_df

In [ ]:
czi_path = r"/Users/rct905/Documents/MaxPlanck_HE_lung_skin/270426_9615_20_skin3a.czi"

czi_dir = "/path/to/czi_folder"
czi_paths = [
    os.path.join(czi_dir, f)
    for f in os.listdir(czi_dir)
    if f.endswith(".czi")
]

checkpoint_path = 'path'

results, summary_df = run_batch_inference(
    czi_paths=czi_paths,
    checkpoint_path="path/to/model.pt",
    top_k=10
)

print(summary_df)

In [ ]:
czi_dir = r"/Users/rct905/Documents/MaxPlanck_HE_lung_skin"
checkpoint_path=r"checkpoints/exp2_h-optimus-0/fold_3_auc_0.9601.pt"

czi_paths = [
    os.path.join(czi_dir, f)
    for f in os.listdir(czi_dir)
    if f.endswith(".czi")
]

results, summary_df = run_batch_inference(
    czi_paths=czi_paths,
    checkpoint_path=checkpoint_path,
    top_k=10
)

print(summary_df)